In [17]:
import hashlib
import random
import json

# Step 1: Create random users and balances
users = []

for i in range(10):  # 10 users for example
    user_id = f"User{i+1:03}"
    balance = round(random.uniform(0.01, 5.0), 2)  # Random balance between 0.01 and 5.0 BTC
    users.append((user_id, balance))

with open('proof_of_reserves_users.json', 'wt') as w:
    w.write(json.dumps(users, indent=2))


In [18]:
users = []
with open('proof_of_reserves_users.json') as r:
    users = json.loads(r.read())

# Print users and balances
print("Users and Balances:")
for user, balance in users:
    print(f"{user}: {balance} BTC")


Users and Balances:
User001: 4.61 BTC
User002: 1.9 BTC
User003: 3.91 BTC
User004: 2.82 BTC
User005: 4.94 BTC
User006: 4.03 BTC
User007: 2.86 BTC
User008: 3.95 BTC
User009: 4.49 BTC
User010: 2.29 BTC


In [19]:
# Step 2: Hash each (user + balance) to create leaves
def hash_leaf(user, balance):
    data = f"{user}{balance}".encode('utf-8')
    return hashlib.sha256(data).hexdigest()

leaves = [hash_leaf(user, balance) for user, balance in users]
leaves

['3257b101b362eda08ce1a365f9f1476f21328927975ad33877280e46c3e95e56',
 'c821143018fe2ee655c1c234c6a3786d0e490f98dbb7bad453bf0ba83f6376d1',
 '61f0500477d90639dcc5b9c262c71ecf66fb266b1a9f2af4022ad5f3d21d7565',
 'c0fe170861f2b6cb4812a37861f98003ac6521569b829f2864f2bf8c3a86aa76',
 '4c7d9a51eb54334f52313e3369a88ae40bead9c2e5a4a994737e35158d58d189',
 '7ff354b795e29376a9a8517801ba286111a5ca0a1858ce843238a7e1fd14a5d3',
 '40742568f56df5cd722e777a8573dfd7c5b88abd641715b5d4fd3d76f7518ffd',
 'd365633d3e32bdcfa4d431a9a23722ed9b3c065e26aadd83a0058290f419972e',
 'b8a73fc8b0014548d56638f88c90e27af8aa69523e66eecd3efa60ef86147cc9',
 '113cccf4c8abb599895ef5d23ab70a1a2e16d296c29616c1485c5a48077ce436']

In [28]:
def build_merkle_tree(leaves):
    tree = [leaves]
    while len(tree[-1]) > 1:
        level = tree[-1]
        next_level = []
        for i in range(0, len(level), 2):
            if i + 1 < len(level):
                h1, h2 = level[i], level[i+1]
            else:
                h1, h2 = level[i], level[i]  # duplicate last if odd
            # Sort before combining!
            if h1 < h2:
                combined = (h1 + h2).encode('utf-8')
            else:
                combined = (h2 + h1).encode('utf-8')
            parent = hashlib.sha256(combined).hexdigest()
            next_level.append(parent)
        tree.append(next_level)
    return tree

merkle_tree = build_merkle_tree(leaves)

merkle_tree

[['3257b101b362eda08ce1a365f9f1476f21328927975ad33877280e46c3e95e56',
  'c821143018fe2ee655c1c234c6a3786d0e490f98dbb7bad453bf0ba83f6376d1',
  '61f0500477d90639dcc5b9c262c71ecf66fb266b1a9f2af4022ad5f3d21d7565',
  'c0fe170861f2b6cb4812a37861f98003ac6521569b829f2864f2bf8c3a86aa76',
  '4c7d9a51eb54334f52313e3369a88ae40bead9c2e5a4a994737e35158d58d189',
  '7ff354b795e29376a9a8517801ba286111a5ca0a1858ce843238a7e1fd14a5d3',
  '40742568f56df5cd722e777a8573dfd7c5b88abd641715b5d4fd3d76f7518ffd',
  'd365633d3e32bdcfa4d431a9a23722ed9b3c065e26aadd83a0058290f419972e',
  'b8a73fc8b0014548d56638f88c90e27af8aa69523e66eecd3efa60ef86147cc9',
  '113cccf4c8abb599895ef5d23ab70a1a2e16d296c29616c1485c5a48077ce436'],
 ['71d38b2eb410deb71f7727efde973940e59c9fb55ee2716332503da3633a83b0',
  'd9c4f93d5d6130c5b8b95e85d4140f7d60d07b34921a882b527e9bff7e8019e4',
  '0b002d81d9a6dbb404f4e381a612412bfe968c714d648ea2e6f4f06799f20495',
  '7e1882957044008deb0c2c99dbc5b6220d444542d783c77e3fc1fe8bd621277a',
  'd3f7dba09f24db55

In [29]:
# Step 4: Merkle Root
merkle_root = merkle_tree[-1][0]
print("\nMerkle Root:", merkle_root)


Merkle Root: c52db21953ae427475e9cb60b6edb3530750d2ad0edd9359fde012ebd7783758


In [30]:
# Step 5: Generate proof for a given user
def get_merkle_proof(index, tree):
    proof = []
    for level in tree[:-1]:  # Exclude root level
        if index % 2 == 0:
            sibling_index = index + 1
        else:
            sibling_index = index - 1
        if sibling_index < len(level):
            proof.append(level[sibling_index])
        else:
            proof.append(level[index])  # Duplicate if no sibling
        index = index // 2
    return proof

# Example: Proof for User005
user_index = 4  # 0-based index; User005
proof = get_merkle_proof(user_index, merkle_tree)

print(f"\nMerkle Proof for {users[user_index][0]} ({users[user_index][1]} BTC):")
for p in proof:
    print(p)




Merkle Proof for User005 (4.94 BTC):
7ff354b795e29376a9a8517801ba286111a5ca0a1858ce843238a7e1fd14a5d3
7e1882957044008deb0c2c99dbc5b6220d444542d783c77e3fc1fe8bd621277a
c364b03b29f2e6d04f0765d67d00ec275cf70f59d4290195141642041006a6c4
a4a2b76b718eb114a939f39a9d1a2f601e5e5de7df677ae9cb55a473366b644f


In [31]:
def verify_proof(leaf_hash, proof, merkle_root):
    computed_hash = leaf_hash
    for sibling_hash in proof:
        # Assume the smaller hash goes first to avoid ambiguity
        if computed_hash < sibling_hash:
            combined = (computed_hash + sibling_hash).encode('utf-8')
        else:
            combined = (sibling_hash + computed_hash).encode('utf-8')
        computed_hash = hashlib.sha256(combined).hexdigest()
    return computed_hash == merkle_root



In [32]:
# Step 6: Verify the proof for User005
leaf = leaves[user_index]

is_valid = verify_proof(leaf, proof, merkle_root)

print(f"\nIs the Merkle Proof valid for {users[user_index][0]}? {is_valid}")



Is the Merkle Proof valid for User005? True
